### Задание

1. Выбрать QA датасет (https://huggingface.co/datasets). 
2. На выбранном датасете обучить модель https://huggingface.co/intfloat/multilingual-e5-small с помощью этого кода.
3. Переписать код под актуальную версию sentence-transformers.
4. Сравнить результаты

In [1]:
import pickle
from pprint import pp
from random import shuffle

from datasets import load_dataset, Dataset
from sentence_transformers import InputExample

/home/ivan/workspace/conversational-ai-and-llms/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch

device = 'mps' if torch.backends.mps.is_built() else 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

cuda


In [3]:
from sentence_transformers import SentenceTransformer, evaluation, models


def raw_bi_encoder():
    word_embedding_model = models.Transformer('intfloat/multilingual-e5-small', max_seq_length=256)
    pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())

    bi_encoder = SentenceTransformer(
        modules=[word_embedding_model, pooling_model],
        device=device,
    )

    return bi_encoder

In [4]:
bi_encoder = raw_bi_encoder()

print(bi_encoder)

/home/ivan/workspace/conversational-ai-and-llms/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


## Служебные функции для работы с данными

In [5]:
def shrink_repeated_samples(
        queries,
        docs,
        labels,
):
    docs_was = set()
    qs = []
    ds = []
    ls = []
    for i in range(len(queries)):
        q, d, l = queries[i], docs[i], labels[i]
        if d in docs_was:
            continue
        qs.append(q)
        ds.append(d)
        ls.append(l)
        docs_was.add(d)

    return qs, ds, ls


def trainval_shuffled_data():
    datafile = 'trainval.pkl'

    good, bad = load_trainval_samples()
    data = good + bad
    shuffle(data)

    with open(datafile, 'wb') as f:
        pickle.dump(data, f)

    return data


def test_shuffled_data():
    datafile = 'test.pkl'

    good, bad = load_test_samples()
    data = good + bad

    with open(datafile, 'wb') as f:
        pickle.dump(data, f)

    return data


def trainval_dataset():
    data = trainval_shuffled_data()

    questions, documents, labels = zip(*data)

    return Dataset.from_dict({
        "question": list(questions),
        "document": list(documents),
        "label": list(labels),
    })


def load_test_samples():
    _, _, test = _load_dataset()

    questions = test['question']
    contexts = test['context']

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def load_trainval_samples():
    train, _, _ = _load_dataset()

    questions = train['question']
    contexts = train['context']

    return _inflate_with_negative_samples(
        questions=questions,
        contexts=contexts,
    )


def _inflate_with_negative_samples(
        questions,
        contexts,
        delta=30,
):
    good_q_c_cos = list(zip(questions, contexts, [1.0] * len(questions)))

    bad_q_c_cos = []
    n = len(good_q_c_cos)
    for i in range(n):
        cur_q, cur_c, _ = good_q_c_cos[i]
        next_q, next_c, _ = good_q_c_cos[(i + delta) % n]
        if next_c != cur_c:
            bad_q_c_cos.append((cur_q, next_c, 0.0))

    return good_q_c_cos, bad_q_c_cos


def _load_raw_dataset():
    dataset = 'dataset.pkl'
    
    ds = load_dataset('virattt/financial-qa-10K')
    ds = ds["train"]

    with open(dataset, 'wb') as f:
        pickle.dump(ds, f)

    return ds


def _load_dataset():
    # dataset = Dataset.from_dict(_load_raw_dataset())
    dataset = _load_raw_dataset()

    split = dataset.train_test_split(test_size=0.4, seed=42)
    val_test = split["test"].train_test_split(test_size=0.5, seed=42)

    train = split["train"]
    validation = val_test["train"]
    test = val_test["test"]

    return train, validation, test

# Метрики близости без дообучения на trainval

In [6]:
trainval = trainval_shuffled_data()

# questions, document_contexts, coss = zip(*trainval[:n_samples])
questions, document_contexts, coss = zip(*trainval)

In [7]:
evaluator = evaluation.EmbeddingSimilarityEvaluator(questions, document_contexts, coss)

In [8]:
bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.8909729238993549,
 'spearman_cosine': 0.8540814762292023,
 'pearson_manhattan': 0.8851883626268142,
 'spearman_manhattan': 0.855096895849645,
 'pearson_euclidean': 0.8857300124930226,
 'spearman_euclidean': 0.855249579493808,
 'pearson_dot': 0.7866868242559583,
 'spearman_dot': 0.7945205786558398,
 'pearson_max': 0.8909729238993549,
 'spearman_max': 0.855249579493808}

# Метрики близости без дообучения на test

In [9]:
test = test_shuffled_data()
test_questions, test_document_contexts, test_coss = zip(*test)

In [10]:
evaluator = evaluation.EmbeddingSimilarityEvaluator(test_questions, test_document_contexts, test_coss)

In [11]:
bi_encoder = raw_bi_encoder()

/home/ivan/workspace/conversational-ai-and-llms/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.8932972310775644,
 'spearman_cosine': 0.8543482546268689,
 'pearson_manhattan': 0.8872242593035118,
 'spearman_manhattan': 0.855466576395344,
 'pearson_euclidean': 0.8878487993019817,
 'spearman_euclidean': 0.855494412694136,
 'pearson_dot': 0.7956703943827776,
 'spearman_dot': 0.8034414225897305,
 'pearson_max': 0.8932972310775644,
 'spearman_max': 0.855494412694136}

# Дообучение

In [13]:
train_dataset = trainval_dataset()
# print(train_dataset[0].__dict__)

In [14]:
type(train_dataset)

datasets.arrow_dataset.Dataset

In [15]:
from sentence_transformers import losses

train_loss = losses.CosineSimilarityLoss(bi_encoder)

In [16]:
train_loss

CosineSimilarityLoss(
  (model): SentenceTransformer(
    (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
    (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  )
  (loss_fct): MSELoss()
  (cos_score_transformation): Identity()
)

In [17]:
# (query_batch, context_batch), labels = next(iter(train_dataset))
# query_batch['input_ids'].shape, context_batch['input_ids'].shape, labels.shape

In [18]:
from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)

args = SentenceTransformerTrainingArguments(
    output_dir="qa/results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_strategy="epoch",
    save_total_limit=2,1
)

trainer = SentenceTransformerTrainer(
    model=bi_encoder,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator,
)

trainer.train()
trainer.save_model("qa/results/final")

Step,Training Loss
500,0.055400
1000,0.042900
1500,0.042000
2000,0.041600
2500,0.025000
3000,0.023000
3500,0.022800
4000,0.023600
4500,0.018500
5000,0.011400


# Загрузка чекпоинта

In [45]:
from sentence_transformers import SentenceTransformer

finetuned_bi_encoder = SentenceTransformer('qa/results')

In [46]:
finetuned_bi_encoder.evaluate(evaluator)

{'pearson_cosine': 0.9388662695071932,
 'spearman_cosine': 0.8608372595565937,
 'pearson_manhattan': 0.9042189262758911,
 'spearman_manhattan': 0.858906376262768,
 'pearson_euclidean': 0.9044034541043204,
 'spearman_euclidean': 0.8588710284022237,
 'pearson_dot': 0.9293298516311845,
 'spearman_dot': 0.8606437293774871,
 'pearson_max': 0.9388662695071932,
 'spearman_max': 0.8608372595565937}

# Расчёт метрик

In [47]:
data = test_shuffled_data()

In [48]:
n_test_samples = 1000
queries, docs, labels = zip(*data[:n_test_samples])

In [49]:
print('docs count before shrinking: ', len(docs))

docs count before shrinking:  1000


In [50]:
docs = list(set(docs))

In [51]:
print('docs count without duplicates: ', len(docs))

docs count without duplicates:  990


In [52]:
doc_embs = finetuned_bi_encoder.encode(
    docs,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches:  13%|█▎        | 4/31 [00:00<00:05,  5.31it/s]

Batches: 100%|██████████| 31/31 [00:02<00:00, 11.15it/s]


In [53]:
query_embs = finetuned_bi_encoder.encode(
    queries,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 32/32 [00:01<00:00, 31.31it/s]


In [54]:
from sentence_transformers import util

In [55]:
cos_sim_score = util.semantic_search(query_embs, doc_embs, top_k=3)

In [56]:
for_first_query = cos_sim_score[0]
print(for_first_query)

[{'corpus_id': 947, 'score': 0.8349486589431763}, {'corpus_id': 65, 'score': 0.604506254196167}, {'corpus_id': 801, 'score': 0.5749332308769226}]


In [57]:
print('Question: ', queries[0])
print()
for no, ir in enumerate(cos_sim_score[0]):
    corpus_id = ir["corpus_id"]
    print('corpus id: ', corpus_id)
    print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
    print()

Question:  How frequently is the investment portfolio of privately held securities assessed for impairment?

corpus id:  947
Document 1: Cosine Similarity is 0.835:

We assess our privately held debt and equity securities strategic investment portfolio quarterly for impairment.

corpus id:  65
Document 2: Cosine Similarity is 0.605:

When an investment declines below cost basis, management evaluates whether the decline in fair value is other than temporary. If deemed other than temporary, an impairment charge is recorded.

corpus id:  801
Document 3: Cosine Similarity is 0.575:

Long-term investments, other than investments made through the Company’s Impact Investment Fund, with original maturities of greater than twelve months but less than 37 months, are classified as available-for-sale and are reported at fair value using the specific identification method. Unrealized gains and losses are primarily excluded from earnings and reported as a component of other comprehensive income (los

In [58]:
data = test_shuffled_data()
n_test_samples = 1000
queries, docs, labels = zip(*data[:n_test_samples])

queries, docs, labels = shrink_repeated_samples(
    queries=queries,
    docs=docs,
    labels=labels,
)
print(len(docs))

990


# Получение предсказаний модели и меток

In [59]:
doc_embs = finetuned_bi_encoder.encode(
    docs,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 31/31 [00:02<00:00, 11.02it/s]


In [60]:
query_embs = finetuned_bi_encoder.encode(
    queries,
    convert_to_tensor=True,
    show_progress_bar=True,
)

Batches: 100%|██████████| 31/31 [00:00<00:00, 31.22it/s]


In [61]:
cos_scores = util.semantic_search(query_embs, doc_embs, top_k=100)

In [62]:
print(len(cos_scores))

990


In [63]:
print(cos_scores[0][:5])

[{'corpus_id': 0, 'score': 0.8349486589431763}, {'corpus_id': 921, 'score': 0.604506254196167}, {'corpus_id': 462, 'score': 0.5749332308769226}, {'corpus_id': 58, 'score': 0.5682641267776489}, {'corpus_id': 771, 'score': 0.5669838190078735}]


In [64]:
top_k = 5

In [65]:
print('Question: ', queries[0])
print()
for no, ir in enumerate(cos_scores[0][:top_k]):
    corpus_id = ir["corpus_id"]
    print('corpus id: ', corpus_id)
    print(f'Document {no + 1}: Cosine Similarity is {ir["score"]:.3f}:\n\n{docs[corpus_id]}')
    print()

Question:  How frequently is the investment portfolio of privately held securities assessed for impairment?

corpus id:  0
Document 1: Cosine Similarity is 0.835:

We assess our privately held debt and equity securities strategic investment portfolio quarterly for impairment.

corpus id:  921
Document 2: Cosine Similarity is 0.605:

When an investment declines below cost basis, management evaluates whether the decline in fair value is other than temporary. If deemed other than temporary, an impairment charge is recorded.

corpus id:  462
Document 3: Cosine Similarity is 0.575:

Long-term investments, other than investments made through the Company’s Impact Investment Fund, with original maturities of greater than twelve months but less than 37 months, are classified as available-for-sale and are reported at fair value using the specific identification method. Unrealized gains and losses are primarily excluded from earnings and reported as a component of other comprehensive income (loss

In [66]:
print(docs[2])

For assets under development, assets are grouped and assessed for impairment by estimating the undiscounted cash flows, which include remaining construction costs, over the asset's remaining useful life. If cash flows do not exceed the carrying amount, impairment based on fair value versus carrying value is considered.


In [67]:
n = len(queries)
print(n)

990


In [68]:
def build_target_mask_for_i(i, n):
    target_mask = [False] * n
    target_mask[i] = True

    return target_mask

In [69]:
def build_pred_mask_for_i(
        cos_scores,
        i,
        n,
):
    pred_mask = [0.0] * n
    qi_scores = cos_scores[i]

    for docid_score in qi_scores:
        doc_id = docid_score['corpus_id']
        score = docid_score['score']
        pred_mask[int(doc_id)] = score

    return pred_mask

# Расчёт метрик: Recall@5, MRR, MAP, NDCG@10

In [70]:
from statistics import mean

from torchmetrics.functional.retrieval import (
    retrieval_average_precision,
    retrieval_normalized_dcg,
    retrieval_recall,
    retrieval_reciprocal_rank,
)

r5s = []
mrrs = []
rmaps = []
ndcgs = []
for i in range(n):
    pred = build_pred_mask_for_i(cos_scores, i=i, n=n)
    target = build_target_mask_for_i(i=i, n=n)

    pred = torch.tensor(pred)
    target = torch.tensor(target)
    
    r5 = retrieval_recall(preds=pred, target=target, top_k=5).item()
    mrr = retrieval_reciprocal_rank(preds=pred, target=target, top_k=5).item()
    rmap = retrieval_average_precision(preds=pred, target=target, top_k=5).item()
    ndcg = retrieval_normalized_dcg(preds=pred, target=target, top_k=10).item()
    
    r5s.append(r5)
    mrrs.append(mrr)
    rmaps.append(rmap)
    ndcgs.append(ndcg)

print('mean recall5@891: ', mean(r5s))
print('mean mrr: ', mean(mrrs))
print('mean rmAP: ', mean(rmaps))
print('mean ndcg10@891: ', mean(ndcgs))

mean recall5@891:  0.9292929292929293
mean mrr:  0.8551515154314764
mean rmAP:  0.8551515154314764
mean ndcg10@891:  0.8829303515077841
